# 02 — Static carry benchmark

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR = (ROOT / "../data/raw").resolve()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Research parameters
REBALANCE_FREQUENCY = "ME"
VOLATILITY_WINDOW = 60
ANNUAL_VOLATILITY_TARGET = 0.10
LEVERAGE_CAP = 4.0
LONG_FRACTION = 0.30
SHORT_FRACTION = 0.30
TRANSACTION_COST_BPS = 5.0
MINIMUM_TRAINING_MONTHS = 60
PROBABILITY_THRESHOLD = 0.50
RANDOM_STATE = 42

print("Data directory:", DATA_DIR)

from src.data_loader import read_parquet, build_fx_panels
from src.features import safe_log_return, forward_implied_carry
from src.portfolio import (
    cross_sectional_carry_weights,
    portfolio_returns,
    volatility_target_scalar,
)
from src.evaluation import summary_statistics


Data directory: C:\Users\vidhi\OneDrive\Desktop\Project lab\Summer\FX_Carry_26_Summer_PL\data\raw


In [2]:
EM_SPOT_FORWARD_FILE = "em_fx_spot_forward_wide.parquet"
G10_SPOT_FORWARD_FILE = "g10_fx_spot_forward_wide.parquet"

em = read_parquet(DATA_DIR / EM_SPOT_FORWARD_FILE)
g10 = read_parquet(DATA_DIR / G10_SPOT_FORWARD_FILE)
raw = pd.concat([g10, em], axis=1)

print(raw.shape)
display(pd.Series(raw.columns, name="column").to_frame().head(100))


(5094, 483)


,column
0,AUD Curncy__PX_ASK
1,AUD Curncy__PX_BID
2,AUD Curncy__PX_LAST
3,AUD12M Curncy__PX_ASK
4,AUD12M Curncy__PX_BID
...,...
95,GBP12M Curncy__PX_LAST
96,GBP1M Curncy__PX_ASK
97,GBP1M Curncy__PX_BID
98,GBP1M Curncy__PX_LAST


In [3]:
spot, forward, diagnostics = build_fx_panels(raw, minimum_currencies=6)

display(diagnostics)
print("Automatically matched currencies:", spot.columns.tolist())

diagnostics.to_csv(
    OUTPUT_DIR / "fx_field_diagnostics.csv",
    index=False,
)


,currency,spot_column,forward_column,forward_root,invert_to_usd_per_fx,status
0,AUD,AUD Curncy__PX_LAST,AUD1M Curncy__PX_LAST,AUD,False,ok
1,BRL,BRL Curncy__PX_LAST,BCN1M Curncy__PX_LAST,BCN,True,ok
2,CAD,CAD Curncy__PX_LAST,CAD1M Curncy__PX_LAST,CAD,True,ok
3,CHF,CHF Curncy__PX_LAST,CHF1M Curncy__PX_LAST,CHF,True,ok
4,CLP,CLP Curncy__PX_LAST,CLN1M Curncy__PX_LAST,CLN,True,ok
5,CNH,CNH Curncy__PX_LAST,CNH1M Curncy__PX_LAST,CNH,True,ok
6,CNY,CNY Curncy__PX_LAST,CHN1M Curncy__PX_LAST,CHN,True,ok
7,COP,COP Curncy__PX_LAST,None,None,True,missing_1m_forward
8,DKK,DKK Curncy__PX_LAST,DKK1M Curncy__PX_LAST,DKK,True,ok
9,EUR,EUR Curncy__PX_LAST,EUR1M Curncy__PX_LAST,EUR,False,ok


Automatically matched currencies: ['AUD', 'BRL', 'CAD', 'CHF', 'CLP', 'CNH', 'CNY', 'DKK', 'EUR', 'GBP', 'HUF', 'IDR', 'ILS', 'INR', 'JPY', 'MXN', 'MYR', 'NOK', 'NZD', 'PHP', 'PLN', 'SEK', 'SGD', 'THB', 'TRY', 'ZAR']


## Automatic field construction

The loader now:

1. identifies USD currency pairs;
2. distinguishes spot from one-month forward fields;
3. matches both fields by currency;
4. converts all pairs to a common USD-per-foreign-currency convention;
5. excludes ambiguous matches instead of silently guessing.

Review the diagnostic table before interpreting results.


In [4]:
currency_returns = safe_log_return(spot)
carry = forward_implied_carry(spot, forward, tenor_months=1)

formation_carry = carry.resample(REBALANCE_FREQUENCY).last()
formation_returns = currency_returns.reindex(
    formation_carry.index,
    method="ffill",
)

formation_weights = cross_sectional_carry_weights(
    formation_carry,
    formation_returns,
    long_fraction=LONG_FRACTION,
    short_fraction=SHORT_FRACTION,
    vol_window=VOLATILITY_WINDOW,
)

daily_weights = formation_weights.reindex(currency_returns.index).ffill()

gross, net, turnover = portfolio_returns(
    currency_returns,
    daily_weights,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)

scalar = volatility_target_scalar(
    net,
    target=ANNUAL_VOLATILITY_TARGET,
    window=VOLATILITY_WINDOW,
    leverage_cap=LEVERAGE_CAP,
)

static_net = (net * scalar).rename("static_carry_net")

display(summary_statistics(static_net, turnover))

static_net.to_csv(OUTPUT_DIR / "static_carry_returns.csv")
daily_weights.to_parquet(OUTPUT_DIR / "static_carry_weights.parquet")
carry.to_parquet(OUTPUT_DIR / "carry_scores.parquet")


c:\Users\vidhi\anaconda3\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
c:\Users\vidhi\anaconda3\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


annual_return          -0.057482
annual_volatility       0.095023
sharpe                 -0.604928
sortino                -0.770934
maximum_drawdown       -0.725463
calmar                 -0.079235
hit_rate                0.454849
observations         5094.000000
annual_turnover         2.244087
dtype: float64